# Tutorial 130 — Inspect source cards and source-first retrieval

[Tutorial 120](./120_context_expansion.ipynb) combined chunk rankings with
deterministic context expansion. This tutorial adds an **independent source-card
index** for describing and ranking whole sources and demonstrates how
four-channel source-first retrieval combines source-level rankings.

The source-card index complements the chunk-level indexes; it does not replace
them. Source cards help discover relevant sources, but answer evidence and
citations must always come from **original source chunks**.

The first registered card kind, `scientific_paper`, is deliberately
specialized; other kinds can be added later without changing the generic
index. All examples here are deterministic: no model inference, network
download, or project workspace is required.

**Before you begin:** See the [tutorial guide](./README.md) for setup,
learning paths, and workspace safety.


## 1. Prepare KíLín

These setup cells follow the other tutorials. They import the repository
package; they do not load an inference or embedding model.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "kilin").is_dir():
    root = root.parent
print(f"KíLín root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


In [ ]:
import kilin

print(kilin.__name__)


## 2. Inspect the source-card policy and registered kinds

The literature project type enables automatic scientific-paper cards.
The separate generation-model settings default to Qwen3-8B, but **reading
settings does not load any model**. Per-project settings and per-source
overrides are documented in
[Configuration](../docs/configuration.md#source-card-indexing-policy).


In [ ]:
from kilin.project_type import project_type
from kilin.retrievals.source_cards import SOURCE_CARD_KINDS
from kilin.model_settings import default_source_card_generation_settings

literature = project_type("literature")
print("Cards enabled:", literature.source_cards_enabled)
print("Preferred kind:", literature.source_card_kind)
print("Automatic generation:", literature.source_cards_automatic)
print("Registered kinds:", SOURCE_CARD_KINDS.identifiers())
settings = default_source_card_generation_settings("torch")
print("Generation model:", settings.model_id)


## 3. Prepare original passages and a source descriptor

`SourceDescriptor` refers to existing chunk identifiers and does not infer
paper eligibility from the `.pdf` suffix. Here the passages are synthetic
but retain source-relative paths and page-local coordinates.


In [ ]:
from kilin.chunking import Chunk
from kilin.retrievals.source_cards import SourceDescriptor

path = Path("papers/example.pdf")
chunks = (
    Chunk("paper-1", path, "We study particle tracking.",
          1, 2, "pdf_page", None, "Page 1"),
    Chunk("paper-2", path, "Methods use fluorescence microscopy.",
          1, 2, "pdf_page", None, "Page 2"),
    Chunk("paper-3", path, "Results show improved localization.",
          1, 2, "pdf_page", None, "Page 3"),
)
descriptor = SourceDescriptor(
    path, "Particle tracking study", ".pdf",
    tuple(chunk.identifier for chunk in chunks),
)
print(descriptor.path, descriptor.chunk_ids)


## 4. Build the scientific-paper request without running a model

The paper kind selects opening and field-relevant passages. The input
budget covers instructions, metadata and passage text together. The
request asks for one eligibility decision and a consistent JSON card.


In [ ]:
from kilin.retrievals.source_cards.kinds.scientific_paper import (
    card_passages,
    card_prompt,
)

selected = card_passages(
    chunks, descriptor.title, max_input_tokens=5000,
)
system, user_prompt = card_prompt(descriptor.title, selected)
print("Selected passage IDs:", [c.identifier for c in selected])
print("System instruction characters:", len(system))
print("User prompt characters:", len(user_prompt))


## 5. Parse a deterministic example reply

A real indexing session would use its separately configured Qwen3-8B model.
Here we simulate that single JSON reply to inspect the validation
contract without downloading or loading model weights. The cited numbers
are converted into the **actual IDs** of passages shown to the model.


In [ ]:
from hashlib import sha256
import json
from kilin.retrievals.source_cards import CARD_STATUS_VALID
from kilin.retrievals.source_cards.kinds.scientific_paper import (
    card_texts,
    parse_card,
)

reply = json.dumps({
    "applicable": True,
    "reason": "",
    "title": "Particle tracking study",
    "authors": "A. Example and B. Scientist",
    "year": "2026",
    "venue": "Example Journal",
    "identifiers": "",
    "abstract": "The study examines particle tracking [1].",
    "aim": "The authors investigate tracking accuracy [1].",
    "methods": "Fluorescence microscopy is used [2].",
    "data": "",
    "findings": "Localization improves [3].",
    "limitations": "",
})
source_sha = sha256(b"example source bytes").hexdigest()
result = parse_card(
    reply, selected, source=descriptor, source_sha256=source_sha,
    model_id="example-model", generation_settings="demo-only",
)
assert result.status == CARD_STATUS_VALID
for statement in result.card.statements:
    print(statement.field, statement.chunk_ids, statement.text)


## 6. Project the card into search texts

Each valid scientific-paper card yields up to three grouped search texts.
The independent vector index embeds these texts, but they remain
**discovery hints**, not citations or verified quotations.


In [ ]:
groups = card_texts(result.card)
print("Search-text groups:", len(groups))
for number, text in enumerate(groups, start=1):
    print(f"Group {number}: {text!r}")


## 7. Save and load the card content independently of vectors

The real index keeps `source_cards.json` separate from
`source_card_vectors.json`. Here we round-trip only the validated card
record in a temporary directory; no embedding model is started.


In [ ]:
from tempfile import TemporaryDirectory
from kilin.retrievals.source_cards import (
    IndexedSourceCard,
    SourceCardIndex,
)
from kilin.retrievals.source_cards.persistence import (
    load_source_cards,
    save_source_cards,
)

record = IndexedSourceCard(
    result, source_sha, 1, 1, "example-model", "demo-only",
)
cards = SourceCardIndex((record,))
with TemporaryDirectory() as temp:
    output = Path(temp) / "source_cards.json"
    save_source_cards(cards, output)
    restored = load_source_cards(output)
    assert restored == cards
    print("Validated card records:", len(restored.records))


## 8. Inspect four-channel source fusion

The source-first strategy combines separate semantic, lexical, code-graph
and source-card **source rankings**. A card may identify a source whose
passages do not appear near the top of ordinary passage rankings. This
example supplies ranked records directly, without performing embedding
inference or claiming that a generated card proves an answer.


In [ ]:
from kilin.retrievals.strategies.retrieval_semantic import (
    SemanticResult,
)
from kilin.retrievals.source_cards.index import SourceCardMatch
from kilin.retrievals.strategies.retrieval_sources import (
    source_first_search,
)

other = Chunk("other", Path("papers/other.pdf"), "Unrelated text.",
              1, 2, "pdf_page", None, "Page 1")
ranked_sources = source_first_search(
    semantic=(SemanticResult(1.0, other),),
    cards=(SourceCardMatch(0.9, path, "scientific_paper"),),
)
for item in ranked_sources:
    print(item.path, item.channels)
assert path in {item.path for item in ranked_sources}


## 9. Apply the distinction in a real project

During project indexing, missing eligible scientific-paper cards are
generated automatically when enabled. Valid cards are reused, while stale
or missing card vectors can be rebuilt without regenerating valid content.
The project's `source_first` intent limits source and passage counts and
reports incomplete source coverage where applicable.

The GUI exposes per-project card-model and indexing controls plus a
confirmed regeneration action. It does not own source-card validation,
files or inference. See the
[retrieval reference](../docs/retrieval_pipeline.md#four-channel-source-first-retrieval-and-evidence-budgets)
and [model settings](../docs/model_management.md#dedicated-source-card-generation-model).

**Important:** A card-selected paper must still be checked against the
original passages before an answer asserts that it supports a claim.


## Continue the tutorial series

**Previous:** [Tutorial 120 — Expand retrieval deterministically](./120_context_expansion.ipynb)

**Next:** [Tutorial 140 — Chat with a KíLín project](./140_project_chat.ipynb)

You have now seen both passage-level retrieval and source-level discovery.
Tutorial 140 brings these concepts into the persistent project workflow.
